In [2]:
import torch
# 检查CUDA是否可用
print("CUDA available:", torch.cuda.is_available())
# 检查cuDNN是否启用
print("cuDNN enabled:", torch.backends.cudnn.enabled)

CUDA available: True
cuDNN enabled: True


In [2]:
print(torch.linspace(3, 10, steps=5))
print(torch.linspace(1, 5, steps=3))

tensor([ 3.0000,  4.7500,  6.5000,  8.2500, 10.0000])
tensor([1., 3., 5.])


In [4]:
torch.logspace(start=0.1, end=1.0, steps=5), torch.logspace(
    start=2, end=2, steps=1, base=2)

(tensor([ 1.2589,  2.1135,  3.5481,  5.9566, 10.0000]), tensor([4.]))

In [5]:
torch.randperm(9)

tensor([0, 4, 6, 3, 5, 2, 1, 7, 8])

In [8]:
from torch import tensor
torch.bernoulli(tensor([0.3, 0.4, 0.5, 0.6, 0.7]))

tensor([0., 1., 1., 0., 0.])

In [9]:
src = torch.arange(1, 11).reshape((2, 5))
src

tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10]])

In [21]:
index = torch.tensor([[0, 1, 2, 0], [2, 1, 0, 2]])
index

tensor([[0, 1, 2, 0],
        [2, 1, 0, 2]])

In [22]:
torch.zeros(3, 5, dtype=src.dtype).scatter_(1, index, src)

tensor([[4, 2, 3, 0, 0],
        [8, 7, 9, 0, 0],
        [0, 0, 0, 0, 0]])

In [14]:
torch.zeros(3, 5, dtype=src.dtype).scatter_(0, index, src)

tensor([[1, 0, 8, 4, 0],
        [0, 7, 0, 0, 0],
        [6, 0, 3, 9, 0]])

## 计算图

In [5]:

w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)
b = torch.add(w, 1)     # retain_grad()
y = torch.mul(a, b)
b.retain_grad()
y.backward()
print(w.grad)

# 查看叶子结点
print("is_leaf:\n", w.is_leaf, x.is_leaf, a.is_leaf, b.is_leaf, y.is_leaf)
# 查看梯度
print("gradient:\n", w.grad, x.grad, a.grad, b.grad, y.grad)
# 查看 grad_fn
print("grad_fn:\n", w.grad_fn, x.grad_fn, a.grad_fn, b.grad_fn, y.grad_fn)

tensor([5.])
is_leaf:
 True True False False False
gradient:
 tensor([5.]) tensor([2.]) None tensor([3.]) None
grad_fn:
 None None <AddBackward0 object at 0x000001FABBDF2D70> <AddBackward0 object at 0x000001FABBDF27A0> <MulBackward0 object at 0x000001FABBDF0310>


C:\Users\xiaof\AppData\Local\Temp\ipykernel_20044\874620817.py:14: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\cb\pytorch_1000000000000\work\build\aten\src\ATen/core/TensorBody.h:494.)
  print("gradient:\n", w.grad, x.grad, a.grad, b.grad, y.grad)


AutoGrad

调用 `y.backward()` 时：
- **关键点**：PyTorch 默认会累加梯度，而不是覆盖梯度。
- 第一次反向传播后，`w.grad` 已经是 `[5.]`。
- 第二次反向传播会再次计算相同的梯度 `[5.]`，并将其累加到 `w.grad` 上。
- 因此，`w.grad` 变为：
  \[
  w.grad = [5.] + [5.] = [10.]
  \]

#### 5. 输出结果
- 第一次 `print(w.grad)` 输出 `[5.]`。
- 第二次 `print(w.grad)` 输出 `[10.]`。

### 总结
第二次调用 `y.backward()` 后，`w.grad` 的结果是 `10`，因为 PyTorch 默认会累加梯度。第一次反向传播计算出的梯度是 `[5.]`，第二次反向传播再次计算出相同的梯度 `[5.]`，并将其累加到 `w.grad` 上，最终结果为 `[10.]`。


In [6]:
# retain_graph=True
import torch
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)
b = torch.add(w, 1)
y = torch.mul(a, b)

y.backward(retain_graph=True)
print(w.grad)
y.backward()
print(w.grad)

tensor([5.])
tensor([10.])


In [7]:
# retain_graph=False
import torch
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)
b = torch.add(w, 1)
y = torch.mul(a, b)

y.backward()
print(w.grad)
y.backward()
print(w.grad)

tensor([5.])


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [8]:
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)
b = torch.add(w, 1)

y0 = torch.mul(a, b)    # y0 = (x+w) * (w+1)    dy0/dw = 2w + x + 1
y1 = torch.add(a, b)    # y1 = (x+w) + (w+1)    dy1/dw = 2

loss = torch.cat([y0, y1], dim=0)       # [y0, y1]
grad_tensors = torch.tensor([1., 2.])
loss.backward(gradient=grad_tensors)
# w =  1* (dy0/dw)  +   2*(dy1/dw)
print(w.grad)

tensor([9.])


In [9]:

x = torch.tensor([3.], requires_grad=True)
y = torch.pow(x, 2)     # y = x**2

# 一阶导数
# grad_1 = dy/dx = 2x = 2 * 3 = 6
grad_1 = torch.autograd.grad(y, x, create_graph=True)
print(grad_1)

# 二阶导数
# grad_2 = d(dy/dx)/dx = d(2x)/dx = 2
grad_2 = torch.autograd.grad(grad_1[0], x)
print(grad_2)

(tensor([6.], grad_fn=<MulBackward0>),)
(tensor([2.]),)


实现自定义Function

In [15]:
from torch.autograd.function import Function


class Exp(Function):
    @staticmethod
    def forward(ctx, i):
        # ============== step1: 函数功能实现 ==============
        result = i.exp()
        print("i:", i)
        # ============== step2: 结果保存，用于反向传播 ==============
        ctx.save_for_backward(result)
        print("res:", i)
        return result

    @staticmethod
    def backward(ctx, grad_output):
        print("grad_output:", grad_output)
        print("ctx-save:", ctx.saved_tensors)
        # ============== step1: 取出结果，用于反向传播 ==============
        result, = ctx.saved_tensors
        # ============== step2: 反向传播公式实现 ==============
        grad_results = grad_output * result
        return grad_results


x = torch.tensor([1.], requires_grad=True)
# 需要使用apply方法调用自定义autograd function
y = Exp.apply(3 * x)
print(y)  # y = e^x = e^1 = 2.7183
y.backward()
# 反传梯度,  x.grad = dy/dx = e^x = e^1  = 2.7183
print(x.grad)

# 关于本例子更详细解释，推荐阅读 https://zhuanlan.zhihu.com/p/321449610

i: tensor([3.], grad_fn=<MulBackward0>)
res: tensor([3.], grad_fn=<MulBackward0>)
tensor([20.0855], grad_fn=<ExpBackward>)
grad_output: tensor([1.])
ctx-save: (tensor([20.0855], grad_fn=<ExpBackward>),)
tensor([60.2566])
